In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
import shutil
import torch.nn as nn
import os
import pandas as pd
import random
import torch.nn.functional as F

from PatchDataset import load_dataset
from torch.utils.data import DataLoader,Subset,Dataset
from torchvision import datasets, transforms
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report, confusion_matrix

from PatchDataset import load_dataset,load_augmented_dataset
from pipeline import fit_binary_classifier, evaluate, set_seed,predict_probs

from torch.optim.lr_scheduler import StepLR
from torch.optim import Adam


In [ ]:
PATCH_SIZE = 224
BATCH_SIZE = 32
SEED = 42

PATCHES_ROOT = "patches_dataset/patches_v3_seed42_pad1.6_iouth0.09"
traincsv_path = os.path.join(PATCHES_ROOT, "metadata.csv")
testcsv_path = "patches_dataset/test_patches_v3/test_metadata.csv"

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:
# loading the dataset
traindf = pd.read_csv(traincsv_path)
traindf["patch_id"] = traindf["patch_id"].astype(int)
traindf["label"] = traindf["label"].astype(int)
traindf["type"] = traindf["type"].astype(str)

testdf = pd.read_csv(testcsv_path)
testdf["patch_id"] = testdf["patch_id"].astype(int)
testdf["label"] = testdf["label"].astype(int)
testdf["type"] = testdf["type"].astype(str)

train_dataset,val_dataset,train_loader,val_loader,test_dataset,test_loader = load_dataset(traindf,testdf,PATCHES_ROOT,BATCH_SIZE)

for xb,yb in train_loader:
    print(xb.shape,yb.shape,min(yb),max(yb))
    break

AUG_ROOT = "patches_dataset/patches_v3_train_aug"
PATH_AUG_METADATA = os.path.join(AUG_ROOT, "metadata.csv")

aug_train_dataset,aug_val_dataset,aug_train_loader,aug_val_loader,aug_test_dataset,aug_test_loader = load_augmented_dataset(
    PATCHES_ROOT,
    AUG_ROOT,
    PATH_AUG_METADATA,
    testcsv_path,
    BATCH_SIZE
)

for xb, yb in aug_train_loader:
    print(xb.shape, yb.shape, min(yb), max(yb))
    break


train/val: 15858 3965
torch.Size([32, 3, 224, 224]) torch.Size([32]) tensor(0) tensor(1)


In [ ]:
from transformers import ViTForImageClassification, ViTImageProcessor
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score, average_precision_score

# Load pretrained ViT for binary classification
model_vit = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224-in21k',
    num_labels=2,  # binary: tree/no-tree
    ignore_mismatched_sizes=True  # replace classification head
).to(device)

processor_vit = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [ ]:

# Freeze backbone, train only head (fast start)
for param in model_vit.vit.parameters():
    param.requires_grad = False

# Unfreeze gradually for better results
unfrozen_layers = ['vit.encoder.layer.11', 'classifier']
for name, param in model_vit.named_parameters():
    if any(layer in name for layer in unfrozen_layers):
        param.requires_grad = True

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model_vit.parameters()), lr=2e-5)
criterion = nn.CrossEntropyLoss()
scheduler = CosineAnnealingLR(optimizer, T_max=10)

def train_vit_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device).long()
        
        # ViT expects processed inputs
        inputs = processor_vit(images, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        optimizer.zero_grad()
        outputs = model(**inputs, labels=labels)
        loss = outputs.loss
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), correct / total

def eval_vit(model, loader, device):
    model.eval()
    all_scores, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in tqdm(loader):
            images, labels = images.to(device), labels.to(device).long()
            inputs = processor_vit(images, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=1)
            tree_scores = probs[:, 1].cpu().numpy()  # P(tree|patch)
            
            all_scores.extend(tree_scores)
            all_labels.extend(labels.cpu().numpy())
    
    return np.array(all_scores), np.array(all_labels)

In [7]:

# Train 10 epochs
print("Training ViT...")
best_auc = 0
for epoch in range(10):
    train_loss, train_acc = train_vit_epoch(model_vit, train_loader, optimizer, criterion, device)
    scheduler.step()
    
    scores, labels = eval_vit(model_vit, test_loader, device)
    auc = roc_auc_score(labels, scores)
    ap = average_precision_score(labels, scores)
    
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Acc={train_acc:.3f}, "
          f"Test AUC={auc:.4f}, AP={ap:.4f}")
    
    if auc > best_auc:
        best_auc = auc
        torch.save(model_vit.state_dict(), 'best_vit_tree_classifier.pth')

print(f"Best AUC: {best_auc:.4f}")


Training ViT...


  0%|          | 0/56 [00:08<?, ?it/s]


NameError: name 'F' is not defined

In [ ]:

# Your existing ROC/PR plots work directly:
fpr, tpr, _ = roc_curve(labels, scores)
plt.plot(fpr, tpr, label=f"ViT AUC={auc:.3f}")
plt.legend()
plt.show()


# from scratch 

In [4]:
class PatchEmbedding(nn.Module):
  def __init__(self, d_model, img_size, patch_size, n_channels):
    super().__init__()

    self.d_model = d_model # Dimensionality of Model
    self.img_size = img_size # Image Size
    self.patch_size = patch_size # Patch Size
    self.n_channels = n_channels # Number of Channels

    self.linear_project = nn.Conv2d(self.n_channels, self.d_model, kernel_size=self.patch_size, stride=self.patch_size)

  # B: Batch Size
  # C: Image Channels
  # H: Image Height
  # W: Image Width
  # P_col: Patch Column
  # P_row: Patch Row
  def forward(self, x):
    x = self.linear_project(x) # (B, C, H, W) -> (B, d_model, P_col, P_row)

    x = x.flatten(2) # (B, d_model, P_col, P_row) -> (B, d_model, P)

    x = x.transpose(-2, -1) # (B, d_model, P) -> (B, P, d_model)

    return x

In [5]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_seq_length):
    super().__init__()

    self.cls_token = nn.Parameter(torch.randn(1, 1, d_model)) # Classification Token

    # Creating positional encoding
    pe = torch.zeros(max_seq_length, d_model)

    for pos in range(max_seq_length):
      for i in range(d_model):
        if i % 2 == 0:
          pe[pos][i] = np.sin(pos/(10000 ** (i/d_model)))
        else:
          pe[pos][i] = np.cos(pos/(10000 ** ((i-1)/d_model)))

    self.register_buffer('pe', pe.unsqueeze(0))

  def forward(self, x):
    # Expand to have class token for every image in batch
    tokens_batch = self.cls_token.expand(x.size()[0], -1, -1)

    # Adding class tokens to the beginning of each embedding
    x = torch.cat((tokens_batch,x), dim=1)

    # Add positional encoding to embeddings
    x = x + self.pe

    return x

In [6]:
class AttentionHead(nn.Module):
  def __init__(self, d_model, head_size):
    super().__init__()
    self.head_size = head_size

    self.query = nn.Linear(d_model, head_size)
    self.key = nn.Linear(d_model, head_size)
    self.value = nn.Linear(d_model, head_size)

  def forward(self, x):
    # Obtaining Queries, Keys, and Values
    Q = self.query(x)
    K = self.key(x)
    V = self.value(x)

    # Dot Product of Queries and Keys
    attention = Q @ K.transpose(-2,-1)

    # Scaling
    attention = attention / (self.head_size ** 0.5)

    attention = torch.softmax(attention, dim=-1)

    attention = attention @ V

    return attention

In [7]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, n_heads):
    super().__init__()
    self.head_size = d_model // n_heads

    self.W_o = nn.Linear(d_model, d_model)

    self.heads = nn.ModuleList([AttentionHead(d_model, self.head_size) for _ in range(n_heads)])

  def forward(self, x):
    # Combine attention heads
    out = torch.cat([head(x) for head in self.heads], dim=-1)

    out = self.W_o(out)

    return out

In [8]:
class TransformerEncoder(nn.Module):
  def __init__(self, d_model, n_heads, r_mlp=4):
    super().__init__()
    self.d_model = d_model
    self.n_heads = n_heads

    # Sub-Layer 1 Normalization
    self.ln1 = nn.LayerNorm(d_model)

    # Multi-Head Attention
    self.mha = MultiHeadAttention(d_model, n_heads)

    # Sub-Layer 2 Normalization
    self.ln2 = nn.LayerNorm(d_model)

    # Multilayer Perception
    self.mlp = nn.Sequential(
        nn.Linear(d_model, d_model*r_mlp),
        nn.GELU(),
        nn.Linear(d_model*r_mlp, d_model)
    )

  def forward(self, x):
    # Residual Connection After Sub-Layer 1
    out = x + self.mha(self.ln1(x))

    # Residual Connection After Sub-Layer 2
    out = out + self.mlp(self.ln2(out))

    return out

In [9]:
class VisionTransformer(nn.Module):
  def __init__(self, d_model, n_classes, img_size, patch_size, n_channels, n_heads, n_layers):
    super().__init__()

    assert img_size[0] % patch_size[0] == 0 and img_size[1] % patch_size[1] == 0, "img_size dimensions must be divisible by patch_size dimensions"
    assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

    self.d_model = d_model # Dimensionality of model
    self.n_classes = n_classes # Number of classes
    self.img_size = img_size # Image size
    self.patch_size = patch_size # Patch size
    self.n_channels = n_channels # Number of channels
    self.n_heads = n_heads # Number of attention heads

    self.n_patches = (self.img_size[0] * self.img_size[1]) // (self.patch_size[0] * self.patch_size[1])
    self.max_seq_length = self.n_patches + 1

    self.patch_embedding = PatchEmbedding(self.d_model, self.img_size, self.patch_size, self.n_channels)
    self.positional_encoding = PositionalEncoding( self.d_model, self.max_seq_length)
    self.transformer_encoder = nn.Sequential(*[TransformerEncoder( self.d_model, self.n_heads) for _ in range(n_layers)])

    # Classification MLP
    #self.classifier = nn.Sequential(
    #    nn.Linear(self.d_model, self.n_classes),
    #    nn.Softmax(dim=-1)
    #)
    self.classifier = nn.Linear(self.d_model, self.n_classes)

  def forward(self, images):
    x = self.patch_embedding(images)

    x = self.positional_encoding(x)

    x = self.transformer_encoder(x)

    x = self.classifier(x[:,0]) #logits 

    return x

In [14]:
d_model = 384
n_classes = 2
img_size = (224,224)
patch_size = (16,16)
n_channels = 3
n_heads = 6
n_layers = 6
batch_size = 32
epochs = 20
lr = 2e-5

In [15]:
print(device)
transformer = VisionTransformer(d_model, n_classes, img_size, patch_size, n_channels, n_heads, n_layers).to(device)

cuda


In [16]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(transformer.parameters(), lr=lr)
scheduler = StepLR(optimizer, step_size=5, gamma=0.5)
EPOCHS = 20
early_stopping = 3

config = {
    "model": "vit_v2",
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": 2e-5,
    "early_stopping_patience": early_stopping,
}

run_dir = "runs/vit_v2_seed42"

run_info, best_model = fit_binary_classifier(
    model=transformer,
    train_loader=train_loader,
    val_loader=val_loader,          
    test_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    run_dir=run_dir,
    config=config,
    scheduler=scheduler,
    patience=early_stopping
)


Model improved
Epoch 001 | train loss 0.5341 f1 0.8363 acc 0.7536 | val loss 0.4386 f1 0.8684 acc 0.7995 | 
Model improved
Epoch 002 | train loss 0.4136 f1 0.8764 acc 0.8206 | val loss 0.3290 f1 0.9024 acc 0.8615 | 
patience 1/3
Epoch 003 | train loss 0.3375 f1 0.9013 acc 0.8604 | val loss 0.3868 f1 0.8698 acc 0.8340 | 
Model improved
Epoch 004 | train loss 0.2934 f1 0.9132 acc 0.8783 | val loss 0.2607 f1 0.9248 acc 0.8941 | 
Model improved
Epoch 005 | train loss 0.2704 f1 0.9199 acc 0.8880 | val loss 0.2459 f1 0.9289 acc 0.8984 | 
Model improved
Epoch 006 | train loss 0.2343 f1 0.9322 acc 0.9055 | val loss 0.2209 f1 0.9393 acc 0.9145 | 
patience 1/3
Epoch 007 | train loss 0.2281 f1 0.9335 acc 0.9076 | val loss 0.2307 f1 0.9348 acc 0.9122 | 
Model improved
Epoch 008 | train loss 0.2169 f1 0.9365 acc 0.9118 | val loss 0.2148 f1 0.9424 acc 0.9185 | 
patience 1/3
Epoch 009 | train loss 0.2063 f1 0.9410 acc 0.9181 | val loss 0.2193 f1 0.9398 acc 0.9150 | 
Model improved
Epoch 010 | train l

In [ ]:
optimizer = Adam(transformer.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):

  training_loss = 0.0
  for i, data in enumerate(train_loader, 0):
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    optimizer.zero_grad()

    outputs = transformer(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    training_loss += loss.item()

  print(f'Epoch {epoch + 1}/{epochs} loss: {training_loss  / len(train_loader) :.3f}')

In [ ]:
correct = 0
total = 0

with torch.no_grad():
  for data in test_loader:
    images, labels = data
    images, labels = images.to(device), labels.to(device)

    outputs = transformer(images)

    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
  print(f'\nModel Accuracy: {100 * correct // total} %')    